In [1]:
!pip install librosa lightgbm optuna pyloudnorm --quiet

In [2]:
import os
import subprocess
import warnings
import kagglehub
import pickle
import numpy as np
import librosa
import pyloudnorm as pyln
import scipy.signal as sig
from scipy.stats import kurtosis, skew

warnings.filterwarnings('ignore')

In [3]:
SAMPLE_RATE = 16000
CLIP_DURATION = 3
TARGET_LEN = SAMPLE_RATE * CLIP_DURATION
N_MFCC = 13
N_MELS = 40
N_MFCC_QUARTERS = 4
TARGET_LUFS = -23.0
STEP_SIZE = 1.0  
RESP_LOW_HZ = 80         
RESP_HIGH_HZ = 2500         
SILENCE_THRESHOLD = 0.002 
CONFIDENCE_HIGH = 0.75         
CONFIDENCE_LOW = 0.55  

In [4]:
path = kagglehub.dataset_download("shafayatulislam/real-audio-data")
print("Dataset path:", path)

ORIG_AUDIO_PATH = "/kaggle/input/datasets/shafayatulislam/real-audio-data/16 Apr 11.07pm_.m4a"
WAV_AUDIO_PATH  = "/kaggle/working/test_audio.wav"

result = subprocess.run(
    ["ffmpeg", "-y", "-i", ORIG_AUDIO_PATH,
     "-ar", str(SAMPLE_RATE), "-ac", "1", WAV_AUDIO_PATH],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("ffmpeg conversion failed:", result.stderr[-500:])
    TEST_AUDIO_PATH = ORIG_AUDIO_PATH
else:
    TEST_AUDIO_PATH = WAV_AUDIO_PATH

full_audio, _ = librosa.load(TEST_AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
print(f"Loaded : {len(full_audio)} samples  ({len(full_audio)/SAMPLE_RATE:.2f}s total)")
print(f"RMS (raw): {np.sqrt(np.mean(full_audio**2)):.4f}")

Dataset path: /kaggle/input/datasets/shafayatulislam/real-audio-data
Loaded : 68949 samples  (4.31s total)
RMS (raw): 0.0404


In [5]:
def apply_bandpass(audio: np.ndarray,
                   sr: int = SAMPLE_RATE,
                   low_hz: float = RESP_LOW_HZ,
                   high_hz: float = RESP_HIGH_HZ) -> np.ndarray:
    nyq  = sr / 2.0
    low  = max(low_hz  / nyq, 0.001)
    high = min(high_hz / nyq, 0.999)
    b, a = sig.butter(4, [low, high], btype='band')
    return sig.filtfilt(b, a, audio).astype(np.float32)

In [6]:
def spectral_subtraction(audio: np.ndarray,
                         sr: int = SAMPLE_RATE,
                         noise_frames: int = 20) -> np.ndarray:
    n_fft      = 512
    hop_length = 128

    stft      = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    magnitude = np.abs(stft)
    phase     = np.angle(stft)

    frame_energies = np.sum(magnitude ** 2, axis=0)
    quiet_idx      = np.argsort(frame_energies)[:noise_frames]
    noise_estimate = np.mean(magnitude[:, quiet_idx], axis=1, keepdims=True)

    alpha = 2.0    
    beta  = 0.01   
    magnitude_clean = np.maximum(
        magnitude - alpha * noise_estimate,
        beta * magnitude
    )

    stft_clean  = magnitude_clean * np.exp(1j * phase)
    audio_clean = librosa.istft(stft_clean, hop_length=hop_length, length=len(audio))
    return audio_clean.astype(np.float32)

In [7]:
def preprocess_phone_audio(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    print("  [1/3] Bandpass filter (80–2500 Hz)…")
    audio = apply_bandpass(audio, sr)

    print("  [2/3] Spectral subtraction (noise reduction)…")
    audio = spectral_subtraction(audio, sr)

    print("  [3/3] Soft clip…")
    audio = np.clip(audio, -1.0, 1.0).astype(np.float32)

    return audio

In [8]:
rms_before = np.sqrt(np.mean(full_audio ** 2))
full_audio_clean = preprocess_phone_audio(full_audio)
rms_after = np.sqrt(np.mean(full_audio_clean ** 2))
print(f"\nRMS before: {rms_before:.4f}")
print(f"RMS after: {rms_after:.4f}")

  [1/3] Bandpass filter (80–2500 Hz)…
  [2/3] Spectral subtraction (noise reduction)…
  [3/3] Soft clip…

RMS before: 0.0404
RMS after: 0.0343


In [9]:
def normalize_loudness(audio: np.ndarray,
                       sr: int = SAMPLE_RATE,
                       target_lufs: float = TARGET_LUFS) -> np.ndarray:
    meter    = pyln.Meter(sr)
    audio64  = audio.astype(np.float64)
    loudness = meter.integrated_loudness(audio64)
    if not (np.isinf(loudness) or np.isnan(loudness)):
        audio64 = pyln.normalize.loudness(audio64, loudness, target_lufs)
    else:
        print("[warn] Loudness measurement returned inf/nan")
    return np.clip(audio64, -1.0, 1.0).astype(np.float32)

In [10]:
full_audio_clean = normalize_loudness(full_audio_clean)
print(f"Loudness normalised to {TARGET_LUFS} LUFS")
print(f"Final RMS: {np.sqrt(np.mean(full_audio_clean**2)):.4f}")

Loudness normalised to -23.0 LUFS
Final RMS: 0.0519


In [11]:
def extract_features(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    feats = []
    mfcc    = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    d_mfcc  = librosa.feature.delta(mfcc)
    d2_mfcc = librosa.feature.delta(mfcc, order=2)

    n_frames = mfcc.shape[1]
    q_size   = max(1, n_frames // N_MFCC_QUARTERS)

    for matrix in (mfcc, d_mfcc, d2_mfcc):
        for q in range(N_MFCC_QUARTERS):
            seg = matrix[:, q * q_size : (q + 1) * q_size]
            if seg.shape[1] == 0:
                seg = matrix[:, -1:]
            feats += list(np.mean(seg, axis=1))
            feats += list(np.std(seg,  axis=1))

    feats += list(kurtosis(mfcc, axis=1, nan_policy='omit'))
    feats += list(skew(    mfcc, axis=1, nan_policy='omit'))

    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feats += list(np.mean(mel_db, axis=1))
    feats += list(np.std( mel_db, axis=1))

    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
    feats += list(np.mean(contrast, axis=1))
    feats += list(np.std( contrast, axis=1))

    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    feats += list(np.mean(chroma, axis=1))
    feats += list(np.std( chroma, axis=1))

    for feat_fn in (
        lambda: librosa.feature.zero_crossing_rate(y=audio),
        lambda: librosa.feature.spectral_centroid(y=audio, sr=sr),
        lambda: librosa.feature.spectral_rolloff(y=audio,  sr=sr),
        lambda: librosa.feature.spectral_bandwidth(y=audio, sr=sr),
        lambda: librosa.feature.rms(y=audio),
    ):
        v = feat_fn()
        feats += [float(np.mean(v)), float(np.std(v))]

    return np.array(feats, dtype=np.float32)

In [12]:
path = kagglehub.dataset_download("shafayatulislam/trainedmodels")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/shafayatulislam/trainedmodels


In [13]:
def load_model(base_name: str, versions: list = ['v4', 'v3']):
    base = '/kaggle/input/datasets/shafayatulislam/trainedmodels'
    for ver in versions:
        path = os.path.join(base, f'{base_name}_{ver}.pkl')
        if os.path.exists(path):
            with open(path, 'rb') as f:
                obj = pickle.load(f)
            print(f"  Loaded {os.path.basename(path)}")
            return obj
    raise FileNotFoundError(f"No model found for {base_name} in {versions}")

scaler   = load_model('scaler')
encoder  = load_model('encoder')
ensemble = load_model('ensemble')
print("\nAll models loaded. Classes:", list(encoder.classes_))

  Loaded scaler_v4.pkl
  Loaded encoder_v4.pkl
  Loaded ensemble_v4.pkl

All models loaded. Classes: [np.str_('crackle'), np.str_('normal'), np.str_('snore'), np.str_('wheeze')]


In [14]:
def predict_sliding_windows(audio: np.ndarray,
                             sr:    int   = SAMPLE_RATE,
                             clip_len: int = TARGET_LEN,
                             step_sec: float = STEP_SIZE,
                             silence_thr: float = SILENCE_THRESHOLD) -> list:
    step    = int(step_sec * sr)
    results = []

    if len(audio) < clip_len:
        audio = np.pad(audio, (0, clip_len - len(audio)))

    starts = list(range(0, len(audio) - clip_len + 1, step))

    for idx, start in enumerate(starts):
        window = audio[start : start + clip_len].copy()

        rms = float(np.sqrt(np.mean(window ** 2)))
        if rms < silence_thr:
            print(f"  Win {idx+1:>2} ({start/sr:.1f}s–{(start+clip_len)/sr:.1f}s): "
                  f"SILENT (RMS={rms:.4f}) — skipped")
            continue

        window = normalize_loudness(window, sr)

        feats        = extract_features(window, sr)
        feats_scaled = scaler.transform(feats.reshape(1, -1))
        probs        = ensemble.predict_proba(feats_scaled)[0]

        results.append({
            'window_idx': idx + 1,
            'start_sec' : start / sr,
            'end_sec'   : (start + clip_len) / sr,
            'rms'       : rms,
            'probs'     : probs,
        })

    return results

print(f"Running sliding-window prediction "
      f"(window={CLIP_DURATION}s, step={STEP_SIZE}s)…\n")
window_results = predict_sliding_windows(full_audio_clean)
print(f"Total windows analysed : {len(window_results)}")

Running sliding-window prediction (window=3s, step=1.0s)…

Total windows analysed : 2


In [15]:
for r in window_results:
    row = (f"{r['window_idx']:>3}  "
           f"{r['start_sec']:>4.1f}s–{r['end_sec']:>5.1f}s  ")
    for p in r['probs']:
        row += f"{p*100:>7.1f}%"
    predicted = encoder.classes_[np.argmax(r['probs'])]
    conf      = float(np.max(r['probs']))
    if conf >= CONFIDENCE_HIGH:
        flag = "✓"
    elif conf >= CONFIDENCE_LOW:
        flag = "~"
    else:
        flag = "?"
    row += f"  → {predicted.capitalize()} {conf*100:.1f}% {flag}"
    print(row)

  1   0.0s–  3.0s     12.0%   30.6%    2.0%   55.4%  → Wheeze 55.4% ~
  2   1.0s–  4.0s     11.6%   51.2%    2.5%   34.8%  → Normal 51.2% ?


In [16]:
if not window_results:
    print("ERROR: No valid windows found. Check the audio file.")
else:
    all_probs   = np.array([r['probs'] for r in window_results])
    rms_weights = np.array([r['rms']   for r in window_results])
    rms_weights = rms_weights / rms_weights.sum() 

    aggregated  = np.average(all_probs, axis=0, weights=rms_weights)

    for i, cls in enumerate(encoder.classes_):
        print(f"  {cls.capitalize():<8} : {aggregated[i]*100:>5.1f}%")

    max_idx     = int(np.argmax(aggregated))
    max_conf    = float(aggregated[max_idx])
    pred_label  = encoder.classes_[max_idx]

  Crackle  :  11.8%
  Normal   :  42.7%
  Snore    :   2.3%
  Wheeze   :  43.3%


In [17]:
print(f"Audio duration: {len(full_audio_clean)/SAMPLE_RATE:.1f}s")
print(f"Windows used: {len(window_results)}")

Audio duration: 4.3s
Windows used: 2


In [18]:
if max_conf >= CONFIDENCE_HIGH:
    print(f"DETECTED:  {pred_label.capitalize()}  ({max_conf*100:.1f}%)")
elif max_conf >= CONFIDENCE_LOW:
    print(f"LIKELY:  {pred_label.capitalize()}  ({max_conf*100:.1f}%)")
else:
    top2 = np.argsort(aggregated)[::-1][:2]
    print(f"UNCERTAIN:")
    for i in top2:
        print(f"{encoder.classes_[i].capitalize():<8} {aggregated[i]*100:.1f}%")

UNCERTAIN:
Wheeze   43.3%
Normal   42.7%
